In [1]:
from pathlib import Path
import pandas as pd

# Main project folder
BASE = Path.cwd().parent

# Load existing held-out test predictions
saved = pd.read_csv(
    BASE / "data" / "processed" / "test_scored.csv"
)

print("Total test leads:", len(saved))
print("\nColumns:", saved.columns.tolist())
print("\nFirst five records:")
print(saved.head())

Total test leads: 1386

Columns: ['actual', 'probability', 'tier']

First five records:
   actual  probability  tier
0       1     0.178947  Cold
1       0     0.175258  Cold
2       1     0.735294   Hot
3       1     0.178947  Cold
4       0     0.430556  Warm


In [2]:
tier_performance = (
    saved.groupby("tier")["actual"]
    .agg(
        Test_Leads="size",
        Actual_Conversions="sum",
        Conversion_Rate="mean"
    )
    .reset_index()
)

tier_performance["Conversion_Rate"] = (
    tier_performance["Conversion_Rate"] * 100
).round(2)

tier_performance = (
    tier_performance
    .set_index("tier")
    .reindex(["Hot", "Warm", "Cold"])
    .reset_index()
)

print(tier_performance.to_string(index=False))

tier  Test_Leads  Actual_Conversions  Conversion_Rate
 Hot         403                 331            82.13
Warm         269                 119            44.24
Cold         714                  84            11.76


In [3]:
output_path = (
    BASE / "data" / "dashboard_ready"
    / "tier_performance_heldout.csv"
)

tier_performance.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: C:\Users\sabit\Documents\Najeeb\New beginning Sep'26\AI_Lead_Revenue_Intelligence\data\dashboard_ready\tier_performance_heldout.csv


In [4]:
import numpy as np

# Total number of held-out test leads
total_leads = len(saved)

# Select the top 20% of leads
top_n = int(np.ceil(total_leads * 0.20))

# Rank leads by predicted conversion probability
top_leads = saved.sort_values(
    "probability", ascending=False, kind="stable"
).head(top_n)

# Actual conversion rates
overall_rate = saved["actual"].mean()
top20_rate = top_leads["actual"].mean()

# Lift calculation
lift_20 = top20_rate / overall_rate

print("Total test leads:", total_leads)
print("Top 20% lead count:", top_n)
print("Overall conversion rate:", round(overall_rate * 100, 2), "%")
print("Top 20% conversion rate:", round(top20_rate * 100, 2), "%")
print("Top-20% Lift:", round(lift_20, 2), "x")

Total test leads: 1386
Top 20% lead count: 278
Overall conversion rate: 38.53 %
Top 20% conversion rate: 87.41 %
Top-20% Lift: 2.27 x


In [5]:
lift_summary = pd.DataFrame({
    "Metric": [
        "Overall Conversion Rate",
        "Top 20% Conversion Rate",
        "Top 20% Lift"
    ],
    "Value": [
        overall_rate * 100,
        top20_rate * 100,
        lift_20
    ]
})

output_path = (
    BASE / "data" / "dashboard_ready"
    / "lift_performance_heldout.csv"
)

lift_summary.to_csv(output_path, index=False)

print(lift_summary.round(2).to_string(index=False))
print("\nSaved to:", output_path)

                 Metric  Value
Overall Conversion Rate  38.53
Top 20% Conversion Rate  87.41
           Top 20% Lift   2.27

Saved to: C:\Users\sabit\Documents\Najeeb\New beginning Sep'26\AI_Lead_Revenue_Intelligence\data\dashboard_ready\lift_performance_heldout.csv


In [6]:
# Editable business assumptions
minutes_per_lead = 10
share_prioritized = 0.20

# Scenario calculations
total_leads = len(saved)
leads_contacted = int(np.ceil(total_leads * share_prioritized))
leads_not_contacted = total_leads - leads_contacted

hours_all_leads = total_leads * minutes_per_lead / 60
hours_prioritized = leads_contacted * minutes_per_lead / 60
hours_avoided = hours_all_leads - hours_prioritized

print("Total leads:", total_leads)
print("Prioritized leads:", leads_contacted)
print("Minutes per lead (assumed):", minutes_per_lead)
print("Hours to qualify all leads:", round(hours_all_leads, 2))
print("Hours to qualify prioritized leads:", round(hours_prioritized, 2))
print("Estimated qualification hours avoided:", round(hours_avoided, 2))

Total leads: 1386
Prioritized leads: 278
Minutes per lead (assumed): 10
Hours to qualify all leads: 231.0
Hours to qualify prioritized leads: 46.33
Estimated qualification hours avoided: 184.67


In [7]:
hours_summary = pd.DataFrame([{
    "Total_Leads": total_leads,
    "Prioritized_Leads": leads_contacted,
    "Minutes_Per_Lead_Assumed": minutes_per_lead,
    "Prioritized_Share": share_prioritized,
    "Hours_All_Leads": round(hours_all_leads, 2),
    "Hours_Prioritized": round(hours_prioritized, 2),
    "Estimated_Hours_Avoided": round(hours_avoided, 2),
    "Scenario_Type": "Illustrative assumption-based estimate"
}])

output_path = (
    BASE / "data" / "dashboard_ready"
    / "qualification_hours_scenario.csv"
)

hours_summary.to_csv(output_path, index=False)

print(hours_summary.to_string(index=False))
print("\nSaved to:", output_path)

 Total_Leads  Prioritized_Leads  Minutes_Per_Lead_Assumed  Prioritized_Share  Hours_All_Leads  Hours_Prioritized  Estimated_Hours_Avoided                          Scenario_Type
        1386                278                        10                0.2            231.0              46.33                   184.67 Illustrative assumption-based estimate

Saved to: C:\Users\sabit\Documents\Najeeb\New beginning Sep'26\AI_Lead_Revenue_Intelligence\data\dashboard_ready\qualification_hours_scenario.csv
